In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, confusion_matrix


In [2]:
df = pd.read_csv("../data/processed/v3/development_v3.csv")

# Fix definitivi contro errori .lower()
df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)
df["source"]  = df["source"].fillna("unknown").astype(str)


In [3]:
df["title_len"] = df["title"].str.len()

df["title_caps_ratio"] = (
	df["title"].str.count(r"[A-Z]") /
	df["title"].str.count(r"[A-Za-z]").clip(lower=1)
)


In [4]:
X = df.drop(columns=["label"])
y = df["label"]

num_cols = [
	"title_len",
	"title_caps_ratio"
]


In [5]:
preprocess = ColumnTransformer(
	transformers=[
		# ARTICLE
		("article_tfidf", TfidfVectorizer(
			max_features=50_000,
			ngram_range=(1, 2),
			min_df=3,
			max_df=0.9,
			stop_words="english",
			sublinear_tf=True
		), "article"),

		# TITLE — più aggressivo
		("title_tfidf", TfidfVectorizer(
			max_features=30_000,
			ngram_range=(1, 3),
			min_df=1,
			max_df=0.9,
			stop_words="english",
			sublinear_tf=True
		), "title"),

		# TITLE STYLE
		("num", StandardScaler(), num_cols),

		# SOURCE
		("source", OneHotEncoder(handle_unknown="ignore"), ["source"])
	],
	n_jobs=-1
)


In [6]:
model = Pipeline([
	("prep", preprocess),
	("clf", LogisticRegression(
		C=1.0,
		max_iter=1000,
		n_jobs=-1
	))
])


In [7]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1s = []
recalls = []
cms = []

for tr, te in skf.split(X, y):
	model.fit(X.iloc[tr], y.iloc[tr])
	yp = model.predict(X.iloc[te])

	f1s.append(f1_score(y.iloc[te], yp, average="macro"))
	recalls.append(recall_score(y.iloc[te], yp, average="macro"))
	cms.append(confusion_matrix(y.iloc[te], yp))

print("STRATEGIA 1A + 1B (article + title++ + source)")
print("Macro F1:", np.mean(f1s))
print("Macro Recall:", np.mean(recalls))
print("Confusion Matrix:\n", np.sum(cms, axis=0))


STRATEGIA 1A + 1B (article + title++ + source)
Macro F1: 0.7007656044478543
Macro Recall: 0.6970448841553993
Confusion Matrix:
 [[18901   638   404   792   197  2385   224]
 [  732  8311   535   359    87   451   113]
 [  674   623  9088   350    49   266   111]
 [ 1649   568   515  4995   608  1432   210]
 [  247    61    15   319  7581   347     4]
 [ 3772   631   298  1165   626  6289   272]
 [  437   153    82   218    35   283  1894]]


In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# =====================
# LOAD
# =====================
df = pd.read_csv("../data/processed/v3/development_v3.csv")

# =====================
# FIX TESTO (PRIMA)
# =====================
df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)
df["source"]  = df["source"].fillna("unknown").astype(str)

# =====================
# SPLIT
# =====================
X = df.drop(columns=["label"])
y = df["label"]

# =====================
# PREPROCESS
# =====================
preprocess_1 = ColumnTransformer(
	transformers=[
		("article", TfidfVectorizer(
			max_features=120_000,
			ngram_range=(1,2),
			min_df=3,
			max_df=0.9,
			sublinear_tf=True,
			stop_words="english"
		), "article"),

		("title", TfidfVectorizer(
			max_features=30_000,
			ngram_range=(1,2),
			min_df=2,
			max_df=0.95,
			sublinear_tf=True,
			stop_words="english"
		), "title"),

		("source", OneHotEncoder(handle_unknown="ignore"), ["source"])
	],
	n_jobs=-1
)

model_1 = Pipeline([
	("prep", preprocess_1),
	("clf", LogisticRegression(
		C=1.0,
		max_iter=1000,
		n_jobs=-1
	))
])

# =====================
# CV
# =====================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1s = []
for tr, te in skf.split(X, y):
	model_1.fit(X.iloc[tr], y.iloc[tr])
	yp = model_1.predict(X.iloc[te])
	f1s.append(f1_score(y.iloc[te], yp, average="macro"))

print("STRATEGY 1 Macro F1:", np.mean(f1s))



STRATEGY 1 Macro F1: 0.7011247131126044


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, confusion_matrix

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("../data/processed/v3/development_v3.csv")

# Fix definitivi contro errori .lower()
df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)
df["source"]  = df["source"].fillna("unknown").astype(str)

X = df.drop(columns=["label"])
y = df["label"]

# =========================
# STRATEGY 2 PREPROCESSOR
# =========================
preprocess_2 = ColumnTransformer(
	transformers=[
		# ARTICLE
		("article", TfidfVectorizer(
			max_features=120_000,
			ngram_range=(1,2),
			min_df=3,
			max_df=0.9,
			sublinear_tf=True,
			stop_words="english"
		), "article"),

		# TITLE
		("title", TfidfVectorizer(
			max_features=30_000,
			ngram_range=(1,2),
			min_df=2,
			max_df=0.95,
			sublinear_tf=True,
			stop_words="english"
		), "title"),

		# SOURCE (categorical semantic bias)
		("source", OneHotEncoder(handle_unknown="ignore"), ["source"]),

		# NUMERIC (only the useful ones)
		("num", StandardScaler(), ["title_ratio", "n_tokens"])
	],
	n_jobs=-1
)

model_2 = Pipeline([
	("prep", preprocess_2),
	("clf", LogisticRegression(
		C=1.0,
		max_iter=1000,
		n_jobs=-1
	))
])

# =========================
# CV + METRICS
# =========================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1s = []
recalls = []
cms = []

for tr, te in skf.split(X, y):
	model_2.fit(X.iloc[tr], y.iloc[tr])
	yp = model_2.predict(X.iloc[te])

	f1s.append(f1_score(y.iloc[te], yp, average="macro"))
	recalls.append(recall_score(y.iloc[te], yp, average="macro"))
	cms.append(confusion_matrix(y.iloc[te], yp))

print("STRATEGY 2 (Semantic + minimal numeric)")
print("Macro F1:", np.mean(f1s))
print("Macro Recall:", np.mean(recalls))
print("Confusion Matrix:\n", np.sum(cms, axis=0))


STRATEGY 2 (Semantic + minimal numeric)
Macro F1: 0.7018251678042141
Macro Recall: 0.6980966594979412
Confusion Matrix:
 [[18975   627   406   784   196  2329   224]
 [  746  8339   525   349    87   431   111]
 [  686   623  9090   345    50   260   107]
 [ 1677   566   514  4974   613  1416   217]
 [  252    55    16   317  7598   332     4]
 [ 3743   640   285  1175   624  6313   273]
 [  453   143    85   226    36   266  1893]]
